# Agri-Link · Crop-quality model benchmark (Colab GPU)

Trains and compares **Custom CNN, VGG16, ResNet50, MobileNetV2, EfficientNetB0** on the *Wild Rice Kernel Quality* dataset (CC-BY-4.0), picks the best by validation macro-F1, reports held-out test metrics, and exports `crop_quality_model_final.keras` + `.json` for `ml_backend/models/`.

**Runtime → Change runtime type → GPU** first. Grade mapping: Premium = healthy; Good = low-severity broken/discoloured; Fair = high-severity broken/discoloured, insect-damaged, unhulled.

## 1. Upload the Agri-Link project zip (`agri-link.zip`) and install

In [ ]:
from google.colab import files
up = files.upload()  # choose agri-link.zip
!unzip -q -o agri-link.zip -d agri-link
!pip -q install pyarrow huggingface_hub scikit-learn matplotlib
%cd agri-link

## 2. Download all 11 raw shards (16,632 kernels, ~5 GB)

In [ ]:
from huggingface_hub import hf_hub_download
import os, shutil
os.makedirs('datasets/raw/wildrice', exist_ok=True)
for i in range(11):
    p = hf_hub_download('Project-AgML/wild_rice_kernel_quality_classification', f'raw/train-{i:05d}-of-00011.parquet', repo_type='dataset')
    shutil.copy(p, f'datasets/raw/wildrice/raw-{i:05d}.parquet')
!ls -la datasets/raw/wildrice

## 3. Build Premium / Good / Fair folders
Increase the caps for the full run (the CPU run used 300 per healthy/minor class, 150 per severe class → 600 per grade).

In [ ]:
!sed -i "s/CAP = {n: (300 if GRADE\[n\] != 'Fair' else 150)/CAP = {n: (1800 if GRADE[n] != 'Fair' else 900)/" datasets/scripts/prepare_wild_rice.py
!rm -rf datasets/quality datasets/demo_quality
!python datasets/scripts/prepare_wild_rice.py datasets/raw/wildrice datasets/quality datasets/demo_quality
!for d in datasets/quality/*; do echo $d $(ls $d | wc -l); done

## 4. Benchmark the five models

In [ ]:
%cd ml_backend
!python benchmark_models.py --data ../datasets/quality --epochs 40 \
  --dataset-name 'Wild Rice Kernel Quality (Univ. of Manitoba) via AgML' --dataset-license CC-BY-4.0 \
  --label-mapping 'Premium=healthy_large+healthy_medium; Good=broken_low+discoloured_low; Fair=broken_high+discoloured_high+insect_infected+unhulled'

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('models/quality_benchmark/BENCHMARK.md').read()))
display(Image('models/quality_benchmark/benchmark_comparison.png'))
display(Image('models/quality_benchmark/confusion_matrices.png'))

## 5. Check the selected model on never-seen kernels and segregate them into Premium / Good / Bad

In [ ]:
!python crop_segregation.py ../datasets/demo_quality | tail -5
!ls ../crop_quality_results/*

## 6. Download the trained model and attach it to the project
Copy both files into `ml_backend/models/` (replacing the CPU-trained ones) and restart the server.

In [ ]:
!cd models && zip -qr ../quality_model.zip crop_quality_model_final.keras crop_quality_model_final.json quality_benchmark
files.download('quality_model.zip')

## 7. Fruits & vegetables quality model
FruitNet (7 Indian fruits, Good/Mixed/Bad, CC-BY-4.0), AgML Tomato quality (CC-BY-4.0) and the apple/banana/orange freshness set (MIT). Premium = good/fresh, Good = FruitNet *Mixed*, Fair = bad/rotten. Raise `CAP` for the full run.

In [ ]:
from huggingface_hub import hf_hub_download
import os, shutil
R='datasets/raw'
for sub in ('fruitnet','tomato','freshness'): os.makedirs(f'{R}/{sub}', exist_ok=True)
for i in range(9):
    shutil.copy(hf_hub_download('Project-AgML/FruitNet_quality_classification', f'data/train-{i:05d}-of-00009.parquet', repo_type='dataset'), f'{R}/fruitnet/p{i}.parquet')
shutil.copy(hf_hub_download('Project-AgML/tomato_quality_classification', 'raw/train-00000-of-00001.parquet', repo_type='dataset'), f'{R}/tomato/raw.parquet')
for f in ('train','test'):
    shutil.copy(hf_hub_download('jojogo9/freshness', f'dataset_100/{f}.parquet', repo_type='dataset'), f'{R}/freshness/{f}.parquet')

In [ ]:
%cd /content/agri-link
!rm -rf datasets/produce datasets/demo_produce
!CAP=400 python datasets/scripts/prepare_produce.py datasets/raw datasets/produce datasets/demo_produce
%cd ml_backend
!python benchmark_models.py --data ../datasets/produce --name produce_quality_model --bench-dir produce_benchmark --epochs 40 \
  --supported-crops 'Apple,Banana,Guava,Lemon,Lime,Orange,Pomegranate,Tomato' \
  --dataset-name 'FruitNet + AgML Tomato quality + jojogo9/freshness' --dataset-license 'CC-BY-4.0 / CC-BY-4.0 / MIT' \
  --label-mapping 'Premium=Good/fresh; Good=FruitNet Mixed; Fair=Bad/rotten'
!python crop_segregation.py ../datasets/demo_produce | tail -3

In [ ]:
!cd models && zip -qr ../produce_model.zip produce_quality_model.json produce_quality_model_head.keras produce_benchmark
files.download('produce_model.zip')